# Création des jeux d'entraînement et de test

Ce notebook lit le fichier `tweets.csv` situé dans le dossier `../data`, effectue un **train-test split stratifié** sur la variable cible, puis exporte :

- `train.csv`
- `test.csv`

dans le même dossier `../data`.

La stratification permet de conserver approximativement la même proportion des classes dans les deux jeux.

In [1]:
# Imports
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

## Paramètres

In [2]:
# Paramètres principaux
NOM_FICHIER_SOURCE = 'tweets.csv'
NOM_FICHIER_TRAIN = 'train.csv'
NOM_FICHIER_TEST = 'test.csv'

COLONNE_CIBLE = 'target'
TAILLE_TEST = 0.20
RANDOM_STATE = 42

## Chargement des données

In [3]:
# Chemins relatifs au notebook situé dans le dossier notebooks/
DOSSIER_DATA = Path('..') / 'data'
CHEMIN_SOURCE = DOSSIER_DATA / NOM_FICHIER_SOURCE
CHEMIN_TRAIN = DOSSIER_DATA / NOM_FICHIER_TRAIN
CHEMIN_TEST = DOSSIER_DATA / NOM_FICHIER_TEST

print(f'Dossier data        : {DOSSIER_DATA.resolve()}')
print(f'Fichier source       : {CHEMIN_SOURCE.resolve()}')
print(f'Fichier train export : {CHEMIN_TRAIN.resolve()}')
print(f'Fichier test export  : {CHEMIN_TEST.resolve()}')

if not CHEMIN_SOURCE.exists():
    raise FileNotFoundError(
        f"Le fichier source est introuvable : {CHEMIN_SOURCE.resolve()}\n"
        "Vérifie que tweets.csv se trouve bien dans le dossier data."
    )

df = pd.read_csv(CHEMIN_SOURCE)

print(f'\nDimensions de la base complète : {df.shape}')
display(df.head())

Dossier data        : C:\Users\DELL\Documents\Classes\ISE2\ISE2_2026\SEM2\ML2\Projet\Disaster-Tweets-NLP\data
Fichier source       : C:\Users\DELL\Documents\Classes\ISE2\ISE2_2026\SEM2\ML2\Projet\Disaster-Tweets-NLP\data\tweets.csv
Fichier train export : C:\Users\DELL\Documents\Classes\ISE2\ISE2_2026\SEM2\ML2\Projet\Disaster-Tweets-NLP\data\train.csv
Fichier test export  : C:\Users\DELL\Documents\Classes\ISE2\ISE2_2026\SEM2\ML2\Projet\Disaster-Tweets-NLP\data\test.csv

Dimensions de la base complète : (11370, 5)


,id,keyword,location,text,target
0,0,ablaze,NaN,"Communal violence in Bhainsa, Telangana. ""Ston...",1
1,1,ablaze,NaN,Telangana: Section 144 has been imposed in Bha...,1
2,2,ablaze,New York City,Arsonist sets cars ablaze at dealership https:...,1
3,3,ablaze,"Morgantown, WV",Arsonist sets cars ablaze at dealership https:...,1
4,4,ablaze,NaN,"""Lord Jesus, your love brings freedom and pard...",0


In [4]:
if COLONNE_CIBLE not in df.columns:
    raise ValueError(
        f"La colonne cible '{COLONNE_CIBLE}' est absente du fichier tweets.csv."
    )

print('Colonnes disponibles :')
print(list(df.columns))

print('\nValeurs manquantes par colonne :')
display(df.isna().sum().to_frame('nb_manquants'))

print('\nRépartition initiale de la cible :')
distribution_initiale = df[COLONNE_CIBLE].value_counts(normalize=True).sort_index().rename('proportion')
display(distribution_initiale.to_frame())

Colonnes disponibles :
['id', 'keyword', 'location', 'text', 'target']

Valeurs manquantes par colonne :


,nb_manquants
id,0
keyword,0
location,3418
text,0
target,0



Répartition initiale de la cible :


,proportion
target,
0,0.814072
1,0.185928


## Split stratifié train / test

In [6]:
train_df, test_df = train_test_split(
    df,
    test_size=TAILLE_TEST,
    random_state=RANDOM_STATE,
    stratify=df[COLONNE_CIBLE]
)

# Réinitialisation des index pour des exports propres
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Taille train : {train_df.shape}")
print(f"Taille test  : {test_df.shape}")

Taille train : (9096, 5)
Taille test  : (2274, 5)


## Contrôle de la répartition des classes après split

In [7]:
distribution_train = train_df[COLONNE_CIBLE].value_counts(normalize=True).sort_index().rename('train')
distribution_test = test_df[COLONNE_CIBLE].value_counts(normalize=True).sort_index().rename('test')
distribution_comparee = pd.concat([distribution_initiale, distribution_train, distribution_test], axis=1)
display(distribution_comparee)

,proportion,train,test
target,,,
0,0.814072,0.814094,0.813984
1,0.185928,0.185906,0.186016


## Export des fichiers CSV

In [8]:
DOSSIER_DATA.mkdir(parents=True, exist_ok=True)

train_df.to_csv(CHEMIN_TRAIN, index=False)
test_df.to_csv(CHEMIN_TEST, index=False)

print('Export terminé avec succès.')
print(f'Train enregistré dans : {CHEMIN_TRAIN.resolve()}')
print(f'Test enregistré dans  : {CHEMIN_TEST.resolve()}')

Export terminé avec succès.
Train enregistré dans : C:\Users\DELL\Documents\Classes\ISE2\ISE2_2026\SEM2\ML2\Projet\Disaster-Tweets-NLP\data\train.csv
Test enregistré dans  : C:\Users\DELL\Documents\Classes\ISE2\ISE2_2026\SEM2\ML2\Projet\Disaster-Tweets-NLP\data\test.csv


## Aperçu final

In [9]:
print('Aperçu du train :')
display(train_df.head())

print('Aperçu du test :')
display(test_df.head())

Aperçu du train :


,id,keyword,location,text,target
0,10497,traumatised,"York, England",Had a dream (nightmare ) last night that I wor...,0
1,1462,body%20bags,Upstate NY,"There it is, on cue! Body Bags Burrow with the...",0
2,997,bleeding,Krypton,it's weird how ppl w vaginas literally spent d...,0
3,6550,hijacking,"Kolkata, India",17 This short urgent thread bcoz ALTIF BUKHARI...,0
4,11165,wounded,"Kuala Lumpur City, Kuala Lumpu",3. Tens of thousands of American soldiers died...,1


Aperçu du test :


,id,keyword,location,text,target
0,611,attacked,UK,#USASupportsTerrorist 15 out of the 19 terrori...,1
1,4178,disaster,fan acc,V commented on a post on WeVerse about the eru...,0
2,6561,hostage,新潟県三条市,Listen to Danrell x Småland - Hostage by HIGH ...,0
3,3125,death,Bahamas,Light Yagami Light Yagami at the start by the ...,0
4,4186,disaster,Hong Kong,"Safe to the evil tyranny, but torture to the c...",0
